# Exercise 4: Analytics with DuckDB

## Objective
Learn how to query CSV files directly with DuckDB, perform joins across multiple files, use GROUP BY aggregations, write window functions, and export results to Parquet.

## Why DuckDB?
DuckDB is an in-process analytical database designed for fast data analysis. Unlike PostgreSQL which requires a running server, DuckDB runs **inside your Python process** and can query CSV and Parquet files directly without importing them first.

**Key differences from psycopg2:**
- No server setup needed: `import duckdb` and you're ready
- Query CSV files directly: `SELECT * FROM 'file.csv'`
- Fast analytical queries (columnar execution)
- Export to Parquet with one command

## Data Files
This exercise uses four CSV files in the `../data/` folder:
- `employees.csv` — 42 employees with salary, hire_date, department, and active status
- `departments.csv` — 6 departments with name, location, and budget
- `products.csv` — 17 products across 4 categories with unit prices
- `sales.csv` — 128 sales records with quantity, date, and region

## Setup: Import DuckDB

Run the cell below to import the required libraries and set up paths.

In [ ]:
import duckdb
import pandas as pd
from pathlib import Path

# Path to the data folder (relative to this notebook)
data_dir = Path("../data")

# Create an in-memory DuckDB connection
con = duckdb.connect()

print("DuckDB is ready. Version:", duckdb.__version__)
print("Data directory:", data_dir.absolute())

---

## Problem 1: Querying CSV Files Directly

### Business Question
Your HR director wants a report on **high-earning employees**. Find all employees with a salary **greater than 90,000**, showing their **first_name**, **last_name**, **salary**, and **hire_date**. Order the results by salary in **descending** order (highest paid first).

**Hint:** DuckDB can query a CSV file directly using `read_csv_auto()`:
```python
con.execute("SELECT * FROM read_csv_auto('path/to/file.csv')").fetchdf()
```

<details>
<summary><b>Click to reveal solution</b></summary>

**Solution:**
```python
query = """
    SELECT first_name, last_name, salary, hire_date
    FROM read_csv_auto('data/employees.csv')
    WHERE salary > 90000
    ORDER BY salary DESC;
"""
con.execute(query).fetchdf()
```

**Explanation:** `read_csv_auto()` reads the CSV file and automatically infers column types (salary becomes DOUBLE, hire_date becomes DATE). The WHERE clause filters for salaries above 90,000, and ORDER BY sorts from highest to lowest.

**Expected output:** 18 rows starting with Alice Chen (145000), Nathan Harris (135000), Amy Adams (130000), down to Bob Martinez (110000) and Victor Wright (95000).
</details>

In [ ]:
# Your solution here:



---

## Problem 2: Joins Across CSV Files

### Business Question
The office manager needs a report showing each **active employee** alongside their **department name** and **location**. Join `employees.csv` with `departments.csv` on the department_id, filter to only active employees (`is_active = TRUE`), and order by department name then last name.

**Hint:** You can reference multiple CSV files in a single query:
```sql
SELECT e.first_name, d.dept_name
FROM read_csv_auto('data/employees.csv') e
JOIN read_csv_auto('data/departments.csv') d ON e.department_id = d.dept_id
```

<details>
<summary><b>Click to reveal solution</b></summary>

**Solution:**
```python
query = """
    SELECT 
        e.first_name,
        e.last_name,
        d.dept_name AS department,
        d.location
    FROM read_csv_auto('data/employees.csv') e
    JOIN read_csv_auto('data/departments.csv') d 
        ON e.department_id = d.dept_id
    WHERE e.is_active = TRUE
    ORDER BY d.dept_name, e.last_name;
"""
con.execute(query).fetchdf()
```

**Explanation:** The INNER JOIN matches employees.department_id with departments.dept_id. This automatically excludes Paula Morris (emp_id 42) who has a NULL department_id. The WHERE clause further filters out inactive employees. Note that `is_active` is read as a BOOLEAN by DuckDB's auto-detection.

**Expected output:** 35 rows showing all active employees with their departments, sorted alphabetically by department then last name.
</details>

In [ ]:
# Your solution here:



---

## Problem 3: GROUP BY with Calculated Revenue

### Business Question
Calculate the **total sales revenue by region**. Revenue for each sale is `quantity * unit_price`. You need to JOIN `sales.csv` with `products.csv` to get the unit_price for each product. Group by region, calculate the total revenue, and order from highest to lowest revenue.

**Note:** Some sales records have an empty region value. Include a row for `''` (empty string) to see revenue for records without a region assigned.

**Hint:** Calculate revenue in the SELECT, then GROUP BY region:
```sql
SELECT region, SUM(s.quantity * p.unit_price) AS total_revenue
FROM read_csv_auto('data/sales.csv') s
JOIN read_csv_auto('data/products.csv') p ON s.product_id = p.product_id
GROUP BY region
```

<details>
<summary><b>Click to reveal solution</b></summary>

**Solution:**
```python
query = """
    SELECT 
        s.region,
        COUNT(*) AS num_sales,
        ROUND(SUM(s.quantity * p.unit_price), 2) AS total_revenue,
        ROUND(AVG(s.quantity * p.unit_price), 2) AS avg_sale_value
    FROM read_csv_auto('data/sales.csv') s
    JOIN read_csv_auto('data/products.csv') p 
        ON s.product_id = p.product_id
    GROUP BY s.region
    ORDER BY total_revenue DESC;
"""
con.execute(query).fetchdf()
```

**Explanation:** The JOIN brings the unit_price from products into each sales row. `SUM(quantity * unit_price)` calculates total revenue per region. We also include `COUNT(*)` and `AVG()` for extra insight. ORDER BY total_revenue DESC puts the highest-revenue region on top.

**Expected output:** 5 rows (North, South, East, West, and empty string). East and West typically lead in revenue due to high-value Hardware sales.
</details>

In [ ]:
# Your solution here:



---

## Problem 4: Window Functions — RANK()

### Business Question
For each **product category**, calculate the **average unit_price**. Then **rank the categories** by their average price from highest to lowest using the `RANK()` window function.

**Hint:** Use a CTE (Common Table Expression) to first compute the averages, then apply RANK() as a window function:
```sql
WITH category_stats AS (
    SELECT category, ROUND(AVG(unit_price), 2) AS avg_price
    FROM read_csv_auto('data/products.csv')
    GROUP BY category
)
SELECT *, RANK() OVER (ORDER BY avg_price DESC) AS price_rank
FROM category_stats;
```

<details>
<summary><b>Click to reveal solution</b></summary>

**Solution:**
```python
query = """
    WITH category_stats AS (
        SELECT 
            category,
            ROUND(AVG(unit_price), 2) AS avg_price,
            COUNT(*) AS product_count
        FROM read_csv_auto('data/products.csv')
        GROUP BY category
    )
    SELECT 
        category,
        product_count,
        avg_price,
        RANK() OVER (ORDER BY avg_price DESC) AS price_rank
    FROM category_stats
    ORDER BY price_rank;
"""
con.execute(query).fetchdf()
```

**Explanation:** The CTE `category_stats` first groups products by category and computes the average price. The outer query then applies `RANK() OVER (ORDER BY avg_price DESC)` to assign a rank to each category. Services ranks #1 (avg ~$11,000) because it includes high-cost consulting packages. Hardware is #2, Training #3, Software #4.

**Expected output:** 4 rows:
| category  | product_count | avg_price  | price_rank |
|-----------|--------------|------------|------------|
| Services  | 5            | ~10900.00  | 1          |
| Hardware  | 5            | ~2839.99   | 2          |
| Training  | 3            | ~3333.33   | 3          |
| Software  | 6            | ~574.99    | 4          |
</details>

In [ ]:
# Your solution here:



---

## Problem 5: Dual Query — PostgreSQL vs DuckDB

### Business Question
Write the **same analytical query** in both PostgreSQL and DuckDB, then verify the results are identical.

**The query:** For each department, count the number of employees and calculate the average salary. Only include departments with **more than 5 employees** (use HAVING). Order by employee count descending.

**Part A — PostgreSQL:** Use the `run_query()` helper (from the psycopg2 notebooks) to query the `company.employees` and `company.departments` tables.

**Part B — DuckDB:** Query the CSV files directly using `read_csv_auto()`.

**Part C — Compare:** Verify that both queries return the same results.

<details>
<summary><b>Click to reveal solution</b></summary>

**Part A — PostgreSQL:**
```python
import psycopg2

def run_query(sql: str) -> pd.DataFrame:
    conn = psycopg2.connect(
        host="localhost", port=5432,
        dbname="week2_db", user="student", password="student123"
    )
    try:
        df = pd.read_sql_query(sql, conn)
        return df
    finally:
        conn.close()

pg_query = """
    SELECT 
        d.dept_name,
        COUNT(e.emp_id) AS employee_count,
        ROUND(AVG(e.salary), 2) AS avg_salary
    FROM company.departments d
    JOIN company.employees e ON d.dept_id = e.department_id
    GROUP BY d.dept_name
    HAVING COUNT(e.emp_id) > 5
    ORDER BY employee_count DESC;
"""

pg_result = run_query(pg_query)
print("=== PostgreSQL Results ===")
display(pg_result)
```

**Part B — DuckDB:**
```python
duckdb_query = """
    SELECT 
        d.dept_name,
        COUNT(e.emp_id) AS employee_count,
        ROUND(AVG(e.salary), 2) AS avg_salary
    FROM read_csv_auto('data/employees.csv') e
    JOIN read_csv_auto('data/departments.csv') d 
        ON e.department_id = d.dept_id
    GROUP BY d.dept_name
    HAVING COUNT(e.emp_id) > 5
    ORDER BY employee_count DESC;
"""

duckdb_result = con.execute(duckdb_query).fetchdf()
print("=== DuckDB Results ===")
display(duckdb_result)
```

**Part C — Compare:**
```python
# Reset index for fair comparison (column order may differ slightly)
pg_sorted = pg_result.sort_values('dept_name').reset_index(drop=True)
dd_sorted = duckdb_result.sort_values('dept_name').reset_index(drop=True)

if pg_sorted[['dept_name', 'employee_count', 'avg_salary']].equals(
    dd_sorted[['dept_name', 'employee_count', 'avg_salary']]
):
    print("Results are IDENTICAL!")
else:
    print("Results differ. Investigating...")
    print("PostgreSQL:\n", pg_sorted)
    print("DuckDB:\n", dd_sorted)
```

**Expected output:** 4 departments with more than 5 employees:
| dept_name     | employee_count | avg_salary   |
|---------------|---------------|--------------|
| Engineering   | 7             | ~95142.86    |
| Sales         | 8             | ~78500.00    |
| Finance       | 6             | ~88333.33    |
| Operations    | 6             | ~75000.00    |

**Why identical?** The CSV files contain the same data that was loaded into PostgreSQL. SQL is SQL — the same query logic works in both engines, only the data source syntax differs (`company.employees` vs `read_csv_auto('data/employees.csv')`).
</details>

In [ ]:
# Part A: PostgreSQL version
import psycopg2

def run_query(sql: str) -> pd.DataFrame:
    """Run a PostgreSQL query and return results as a DataFrame."""
    conn = psycopg2.connect(
        host="localhost", port=5432,
        dbname="week2_db", user="student", password="student123"
    )
    try:
        df = pd.read_sql_query(sql, conn)
        return df
    finally:
        conn.close()

# Your PostgreSQL solution here:
pg_query = """
    
"""
pg_result = run_query(pg_query)
display(pg_result)

In [ ]:
# Part B: DuckDB version
# Your DuckDB solution here:
duckdb_query = """
    
"""
duckdb_result = con.execute(duckdb_query).fetchdf()
display(duckdb_result)

In [ ]:
# Part C: Compare results
# Your comparison code here:



---

## Bonus: Exporting DuckDB Results to Parquet

DuckDB makes it trivial to export query results to Parquet format — a compressed, columnar file format that is much faster to read than CSV and takes up less disk space.

### Method 1: Using the COPY command in SQL

In [ ]:
# Export high-earning employees to Parquet using SQL COPY
export_query = """
    COPY (
        SELECT first_name, last_name, salary, hire_date
        FROM read_csv_auto('data/employees.csv')
        WHERE salary > 90000
        ORDER BY salary DESC
    ) TO 'data_output/high_earners.parquet' (FORMAT PARQUET);
"""
con.execute(export_query)
print("Exported high_earners.parquet")

In [ ]:
# Read it back — much faster than reading CSV!
con.execute("SELECT * FROM read_parquet('data_output/high_earners.parquet')").fetchdf()

### Method 2: Using pandas to_parquet()

In [ ]:
# Export the revenue-by-region query result using pandas
revenue_query = """
    SELECT 
        s.region,
        COUNT(*) AS num_sales,
        ROUND(SUM(s.quantity * p.unit_price), 2) AS total_revenue
    FROM read_csv_auto('data/sales.csv') s
    JOIN read_csv_auto('data/products.csv') p 
        ON s.product_id = p.product_id
    GROUP BY s.region
    ORDER BY total_revenue DESC;
"""

revenue_df = con.execute(revenue_query).fetchdf()
revenue_df.to_parquet('data_output/revenue_by_region.parquet', index=False)
print("Exported revenue_by_region.parquet")

# Verify by reading it back
pd.read_parquet('data_output/revenue_by_region.parquet')

### Parquet vs CSV: Why it matters

| Feature          | CSV          | Parquet        |
|------------------|--------------|----------------|
| File size        | Larger       | 2-10x smaller (compressed) |
| Read speed       | Slower       | Much faster (columnar)     |
| Type safety      | All strings  | Preserves types            |
| Human readable   | Yes          | No (binary format)         |
| DuckDB support   | Yes          | Yes (native)               |

**Rule of thumb:** Use CSV for small, human-editable data. Use Parquet for anything you'll query repeatedly.

---

## Cleanup

In [ ]:
# Close the DuckDB connection
con.close()
print("DuckDB connection closed.")